# ChickenStress — 04. Inference Produksi (Quality Gate -> VLM+LoRA -> Confidence/Temporal Voting)
### Bagian 4 dari 4 (`01_image_processing.ipynb` -> `02_models.ipynb` -> `03_evaluate.ipynb` -> `04_inference.ipynb`)

Berbeda dari draft sebelumnya: notebook ini sekarang memanggil **model VLM+LoRA sungguhan**
(bukan simulasi) di tahap [E], dimuat langsung dari **Hugging Face Hub** -- supaya notebook ini
independen dari sesi Kaggle tempat training berjalan. Komponen [A]/[C]/[D]/[F] (quality gate,
enhancement, skip log, confidence+temporal voting) dipertahankan persis dari draft awal.

> **Repo adapter = private.** Butuh HF access token (read) -- dibaca dari env `HF_TOKEN`, login cache (`huggingface-cli login`), atau prompt `getpass`. Token tidak pernah di-hardcode.


In [1]:
!pip install -q -U transformers peft accelerate
!pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.1 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [3]:
import os
import re
import getpass
import json
import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image
from collections import deque, Counter

# --- Config model (ganti sesuai repo Hugging Face kamu setelah cell "Push ke Hugging Face" di bawah) ---
HF_ADAPTER_REPO = "Arga23/chickenstress-vlm-lora"   # <- ganti sesuai repo_id kamu
BASE_MODEL_NAME = "Qwen/Qwen3-VL-4B-Instruct"          # <- samakan dgn model_name saat training di Kaggle

# Path dataset lokal cuma dipakai untuk DEMO quality-gate (opsional, bisa diganti stream kamera).
DEST = "/kaggle/input/datasets/argamasijabat/raw-data-chicken-stress/Data"   

# --- Auth Hugging Face (repo adapter PRIVATE) ---
from huggingface_hub import HfApi, get_token
from huggingface_hub.utils import RepositoryNotFoundError, GatedRepoError, HfHubHTTPError


def resolve_hf_token():
    """Urutan: env HF_TOKEN -> login cache (huggingface-cli login) -> prompt getpass (tidak tersimpan di notebook)."""
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or get_token()
    if not token:
        token = getpass.getpass("Masukkan HF access token (read): ").strip()
    if not token:
        raise RuntimeError("HF token kosong -- repo private tidak bisa diakses tanpa token.")
    return token


HF_TOKEN = resolve_hf_token()

# Fail-fast: cek akses ke repo private SEKARANG, bukan setelah base model 4B selesai di-download.
try:
    _info = HfApi().model_info(HF_ADAPTER_REPO, token=HF_TOKEN)
    print(f"Akses OK: {HF_ADAPTER_REPO} (private={_info.private})")
except (RepositoryNotFoundError, GatedRepoError):
    raise RuntimeError(
        f"Repo '{HF_ADAPTER_REPO}' tidak ditemukan / token tidak punya akses. Cek: (1) repo_id benar, "
        "(2) token milik akun yang sama / punya akses read, (3) kalau fine-grained token, repo ini sudah "
        "di-whitelist di permission token.")
except HfHubHTTPError as e:
    raise RuntimeError(f"Token ditolak Hugging Face (kemungkinan invalid/expired): {e}")


Akses OK: Arga23/chickenstress-vlm-lora (private=True)


## [A]/[B]/[C]/[D] Image Quality Gate -- persis dari draft awal, tidak diubah.

In [4]:
def quality_score(img_bgr, blur_threshold=100, dark_thresh=40, bright_thresh=215):
    """[A] Quality Scoring -- dijalankan cepat, TANPA deep learning, di edge device
    sebelum citra dikirim ke model klasifikasi."""
    if img_bgr is None:
        return {"ok": False, "issues": ["gagal_dibaca_atau_blank"]}
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    laplacian_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    brightness = float(gray.mean())
    is_blank = bool(gray.std() < 3)
    issues = []
    if laplacian_var < blur_threshold: issues.append("blur")
    if brightness < dark_thresh: issues.append("terlalu_gelap")
    if brightness > bright_thresh: issues.append("terlalu_terang")
    if is_blank: issues.append("blank_or_occluded")
    return {"ok": len(issues) == 0, "laplacian_var": laplacian_var, "brightness": brightness,
            "is_blank": is_blank, "issues": issues}


def enhance_image(img_bgr):
    """[C] Perbaikan ringan: CLAHE + denoising + sharpening -- HANYA di inference."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
    enhanced = cv2.fastNlMeansDenoisingColored(enhanced, None, h=5, hColor=5, templateWindowSize=7, searchWindowSize=21)
    sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    return cv2.filter2D(enhanced, -1, sharpen_kernel)


quality_skip_log = []  # [D]: log frame yang di-skip -- umpan balik ke monitoring kesehatan perangkat IoT

def image_quality_gate(image_path, camera_id="unknown", kandang_id="unknown", blur_threshold=100):
    """Orkestrasi [A] -> [B]/[C] -> [D] untuk satu frame."""
    img_bgr = cv2.imread(image_path)
    score = quality_score(img_bgr, blur_threshold=blur_threshold)
    if score["ok"]:
        return img_bgr, "ok"
    enhanced = enhance_image(img_bgr)
    score_after = quality_score(enhanced, blur_threshold=blur_threshold)
    if score_after["ok"]:
        return enhanced, "enhanced"
    quality_skip_log.append({"image_path": image_path, "camera_id": camera_id, "kandang_id": kandang_id,
                              "issues_before": score["issues"], "issues_after": score_after["issues"]})
    return None, "skipped"

## [E] Model VLM + LoRA -- dimuat dari Hugging Face Hub

Sama persis dengan definisi indikator, prompt, dan aturan SCS di `02_models.ipynb`/`03_evaluate.ipynb`
Kaggle v5/v6 (disalin verbatim supaya konsisten dengan yang dipakai saat training & evaluasi).

In [5]:
# --- 7 indikator SCS + aturan (identik dgn Kaggle 02_models.ipynb) ---
SCS_INDICATORS = [
    {"key": "posisi_ekor", "name": "Posisi Ekor", "question": "Apakah ekor ayam menurun atau terkulai?",
     "stres": "Ekor terlihat menurun atau terkulai.", "tidak_stres": "Ekor tegak / posisi normal."},
    {"key": "posisi_kepala", "name": "Posisi Kepala", "question": "Apakah kepala ayam tertunduk atau ditarik ke dalam?",
     "stres": "Kepala terlihat ditarik ke dalam / menunduk.", "tidak_stres": "Kepala tegak, posisi normal."},
    {"key": "penutupan_mata", "name": "Penutupan Mata", "question": "Apakah mata ayam tertutup sebagian atau seluruhnya?",
     "stres": "Mata tertutup sebagian atau seluruhnya.", "tidak_stres": "Mata terbuka penuh."},
    {"key": "pembukaan_paruh", "name": "Pembukaan Paruh",
     "question": "Apakah paruh ayam terbuka sebagian atau seluruhnya (terutama saat bernapas/terengah-engah)?",
     "stres": "Paruh terbuka sebagian atau seluruhnya, terutama saat bernapas/terengah-engah.",
     "tidak_stres": "Paruh tertutup / normal."},
    {"key": "posisi_sayap", "name": "Posisi Sayap", "question": "Apakah sayap ayam terkulai atau menggantung?",
     "stres": "Sayap terlihat terkulai atau menggantung.", "tidak_stres": "Sayap terlipat rapi di badan."},
    {"key": "postur_kaki", "name": "Postur Kaki",
     "question": "Apakah ayam menunjukkan postur membungkuk/merunduk dengan kaki menekuk, berjongkok, duduk, atau berbaring?",
     "stres": "Postur membungkuk/merunduk, kaki menekuk, berjongkok, duduk, atau berbaring.",
     "tidak_stres": "Berdiri tegak dengan kaki lurus."},
    {"key": "kondisi_bulu", "name": "Kondisi Bulu",
     "question": "Apakah bulu ayam mengembang atau kusut, terutama di area leher, punggung bawah, dan dada bawah?",
     "stres": "Bulu mengembang atau kusut, terutama di leher, punggung bawah, dan dada bawah.",
     "tidak_stres": "Bulu rapi, menempel ke badan."},
]
STRESS_THRESHOLD = 3          # HARUS sama dengan threshold final yang dipakai saat training terakhir di Kaggle
MIN_VISIBLE_INDICATORS = 5    # idem -- lihat `vlm_experiments["_meta"]` di Kaggle kalau lupa nilainya
CLASS_NAMES_3 = ["not_stress", "stress", "undefined"]


def build_scs_system_prompt(indicators=SCS_INDICATORS, threshold=STRESS_THRESHOLD, min_visible=MIN_VISIBLE_INDICATORS):
    lines = [
        "Kamu adalah asisten penilai postur ayam berdasarkan Stressed Chicken Scale (SCS).",
        "Amati SATU ekor ayam pada gambar dari posisi lateral (samping), lalu nilai 7 indikator berikut.",
        "Untuk setiap indikator, jawab salah satu dari TIGA pilihan berikut, HANYA berdasarkan apa yang",
        "benar-benar terlihat di gambar -- jangan menebak, jangan menilai dari asumsi lain:",
        '  - "YA"             -> sinyal stres pada indikator ini MUNCUL dan terlihat jelas.',
        '  - "TIDAK"          -> bagian tubuh terkait TERLIHAT jelas, dan sinyal stresnya TIDAK muncul.',
        '  - "TIDAK_TERLIHAT" -> bagian tubuh terkait TIDAK ADA / TIDAK TERLIHAT di foto -- jangan menebak.',
        "",
    ]
    for i, ind in enumerate(indicators, 1):
        lines.append(f"{i}. {ind['name']} -- {ind['question']}")
        lines.append(f"   YA (stres) jika: {ind['stres']}")
        lines.append(f"   TIDAK jika: {ind['tidak_stres']}")
        lines.append("   TIDAK_TERLIHAT jika bagian tubuh ini tidak tampak sama sekali di foto.")
    lines += [
        "", "Balas HANYA dengan JSON valid, tanpa teks lain, dengan format persis:", "{", '  "indicators": {',
    ] + [f'    "{ind["key"]}": "YA, TIDAK, atau TIDAK_TERLIHAT",' for ind in indicators] + [
        "  },", '  "label": "stress, not_stress, atau undefined"', "}",
    ]
    return "\n".join(lines)


SCS_SYSTEM_PROMPT = build_scs_system_prompt()
SCS_USER_PROMPT = ("Berikut adalah foto satu ekor ayam. Nilai postur ayam ini sesuai 7 indikator "
                    "Stressed Chicken Scale yang sudah dijelaskan, lalu balas dalam format JSON yang diminta.")


def parse_vlm_json(raw_text):
    match = re.search(r"\{.*\}", raw_text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"Tidak ditemukan JSON pada output VLM: {raw_text!r}")
    return json.loads(match.group(0))


def normalize_indicator_answer(raw_value):
    raw_val = str(raw_value).strip().upper().replace(" ", "_")
    if raw_val.startswith("YA"):
        return "YA"
    if raw_val.startswith("TIDAK_TERLIHAT") or raw_val in {"TT", "NA", "N/A", "TIDAK_ADA", "TIDAK_TAMPAK", "NONE", ""}:
        return "TIDAK_TERLIHAT"
    if raw_val.startswith("TIDAK"):
        return "TIDAK"
    return "TIDAK_TERLIHAT"


def apply_scs_rule(indicators_dict, indicators=SCS_INDICATORS, threshold=STRESS_THRESHOLD, min_visible=MIN_VISIBLE_INDICATORS):
    total_ya, visible_count, normalized = 0, 0, {}
    for ind in indicators:
        answer = normalize_indicator_answer(indicators_dict.get(ind["key"], "TIDAK_TERLIHAT"))
        normalized[ind["key"]] = answer
        if answer != "TIDAK_TERLIHAT":
            visible_count += 1
        if answer == "YA":
            total_ya += 1
    if visible_count < min_visible:
        label = "undefined"
    else:
        label = "stress" if total_ya >= threshold else "not_stress"
    return normalized, total_ya, visible_count, label, total_ya / len(indicators)

In [6]:
# --- Muat base model + adapter LoRA LANGSUNG dari Hugging Face Hub (bukan checkpoint lokal Kaggle) ---
from transformers import AutoProcessor
from peft import PeftModel

device = "cuda" if torch.cuda.is_available() else "cpu"
model_dtype = torch.float16 if device == "cuda" else torch.float32


def load_vlm_base_model(model_name):
    """Identik dengan Kaggle: coba beberapa kelas Auto sesuai arsitektur VLM-nya."""
    errors = []
    if "qwen3-vl" in model_name.lower() or "qwen3_vl" in model_name.lower():
        try:
            from transformers import Qwen3VLForConditionalGeneration
            return Qwen3VLForConditionalGeneration.from_pretrained(model_name, dtype=model_dtype, low_cpu_mem_usage=True)
        except Exception as e:
            errors.append(str(e))
    try:
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText.from_pretrained(model_name, dtype=model_dtype, low_cpu_mem_usage=True)
    except Exception as e:
        errors.append(str(e))
    raise ImportError("Gagal load VLM:\n" + "\n".join(errors))


processor = AutoProcessor.from_pretrained(BASE_MODEL_NAME, min_pixels=256 * 28 * 28, max_pixels=512 * 28 * 28)
base_vlm_model = load_vlm_base_model(BASE_MODEL_NAME).to(device)
base_vlm_model.eval()

# PeftModel.from_pretrained menerima repo_id Hugging Face langsung. Karena repo PRIVATE, token= wajib
# (tanpa token -> 401/RepositoryNotFoundError). Base model Qwen publik, jadi tidak butuh token.
finetuned_vlm_model = PeftModel.from_pretrained(base_vlm_model, HF_ADAPTER_REPO, token=HF_TOKEN)
finetuned_vlm_model.eval()
print(f"Adapter LoRA dimuat dari Hugging Face Hub: {HF_ADAPTER_REPO}")

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/713 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/23.6M [00:00<?, ?B/s]

Adapter LoRA dimuat dari Hugging Face Hub: Arga23/chickenstress-vlm-lora


In [7]:
def run_vlm_raw(image_path_or_array, model, proc, system_prompt=SCS_SYSTEM_PROMPT, user_prompt=SCS_USER_PROMPT, max_new_tokens=400):
    """Terima path file ATAU array BGR (hasil image_quality_gate) -- fleksibel utk stream kamera."""
    if isinstance(image_path_or_array, str):
        image = Image.open(image_path_or_array).convert("RGB")
    else:
        image = Image.fromarray(cv2.cvtColor(image_path_or_array, cv2.COLOR_BGR2RGB))
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_prompt}]},
    ]
    chat_text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[chat_text], images=[image], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    return proc.batch_decode(trimmed, skip_special_tokens=True)[0]


def classify_image_with_vlm(image_path_or_array, model=finetuned_vlm_model, proc=processor):
    """[E] Pipeline lengkap satu frame: VLM -> parse JSON -> rule SCS -> label akhir (3 kelas)."""
    raw_text = run_vlm_raw(image_path_or_array, model, proc)
    try:
        parsed = parse_vlm_json(raw_text)
        normalized, total_ya, visible_count, label, score = apply_scs_rule(parsed.get("indicators", {}))
        parse_error = None
    except Exception as e:
        normalized, total_ya, visible_count, label, score = apply_scs_rule({ind["key"]: "TIDAK_TERLIHAT" for ind in SCS_INDICATORS})
        parse_error = str(e)
    return {"indicators": normalized, "total_ya": total_ya, "visible_count": visible_count,
            "label": label, "score_stress": score, "raw_response": raw_text, "parse_error": parse_error}

## [F] Confidence Gate & Voting Temporal

Sekarang dipakai bersama hasil klasifikasi **sungguhan**, bukan disimulasikan seperti draft awal.

In [8]:
def make_temporal_voter(window_size=5, confidence_threshold=0.6, min_votes_for_alert=3):
    """confidence < threshold -> diabaikan. Window penuh -> alarm cuma kalau mayoritas = 'stress'."""
    window = deque(maxlen=window_size)

    def update(label, confidence):
        if label == "undefined":
            return {"trigger_alert": False, "reason": "label_undefined_foto_kurang_informatif"}
        if confidence < confidence_threshold:
            return {"trigger_alert": False, "reason": "confidence_rendah_tunggu_frame_berikutnya"}
        window.append(label)
        if len(window) < window.maxlen:
            return {"trigger_alert": False, "reason": "menunggu_window_temporal_penuh"}
        vote_label, vote_count = Counter(window).most_common(1)[0]
        if vote_label == "stress" and vote_count >= min_votes_for_alert:
            return {"trigger_alert": True, "reason": f"mayoritas {vote_count}/{window.maxlen} frame = stress", "label": vote_label}
        return {"trigger_alert": False, "reason": f"mayoritas belum stress ({vote_count}/{window.maxlen})", "label": vote_label}

    return update


def confidence_from_result(result):
    """label 'stress' -> makin tinggi score_stress makin yakin; 'not_stress' -> kebalikannya."""
    if result["label"] == "stress":
        return result["score_stress"]
    if result["label"] == "not_stress":
        return 1 - result["score_stress"]
    return 0.0  # undefined -> voter akan skip lewat cek label di atas

## Demo end-to-end: Quality Gate -> VLM+LoRA -> Confidence/Temporal Voting

Ganti `demo_files` dengan frame dari stream kamera sungguhan di produksi.

In [9]:
demo_dir = os.path.join(DEST, "test")
demo_files = sorted(
    os.path.join(demo_dir, f) for f in os.listdir(demo_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
)[:15]

voter = make_temporal_voter()
rows = []
for path in demo_files:
    frame, status = image_quality_gate(path, camera_id="cam01", kandang_id="kandang01")
    if frame is None:
        rows.append({"file": os.path.basename(path), "quality_status": status, "label": None, "vote": None})
        continue
    result = classify_image_with_vlm(frame)
    conf = confidence_from_result(result)
    vote = voter(result["label"], conf)
    rows.append({"file": os.path.basename(path), "quality_status": status, "label": result["label"],
                 "confidence": round(conf, 3), "vote_trigger_alert": vote["trigger_alert"], "vote_reason": vote["reason"]})

results_df = pd.DataFrame(rows)
print(f"Frame di-skip quality gate: {len(quality_skip_log)} dari {len(demo_files)}")
display(results_df)

Frame di-skip quality gate: 0 dari 2


,file,quality_status,label,confidence,vote_trigger_alert,vote_reason
0,IMG_20250513_132807_jpg.rf.ff9bad1d002a05b54bb...,ok,stress,0.429,False,confidence_rendah_tunggu_frame_berikutnya
1,IMG_20250514_140208_jpg.rf.a9d5ba02679333519e6...,ok,stress,0.429,False,confidence_rendah_tunggu_frame_berikutnya


## Push adapter LoRA ke Hugging Face Hub

Jalankan sekali dari **Kaggle** (setelah training, sel ini pakai `finetuned_vlm_model` & `processor`
hasil training di `02_models.ipynb`), supaya notebook inferensi ini bisa memuat adapter dari
`HF_ADAPTER_REPO` tanpa bergantung pada file lokal Kaggle yang sesinya sementara.

Perlu token **write** (`huggingface-cli login` / `HF_TOKEN` / Kaggle Secrets; jangan hardcode token di notebook). Repo dibuat `private=True`.

In [10]:
# Jalankan di KAGGLE, bukan di sini -- butuh finetuned_vlm_model & processor hasil training asli.
from kaggle_secrets import UserSecretsClient
from huggingface_hub import create_repo

HF_WRITE_TOKEN = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
HF_ADAPTER_REPO = "Arga23/chickenstress-vlm-lora"

create_repo(HF_ADAPTER_REPO, private=True, exist_ok=True, token=HF_WRITE_TOKEN)
finetuned_vlm_model.push_to_hub(HF_ADAPTER_REPO, private=True, token=HF_WRITE_TOKEN,
                                commit_message="LoRA adapter chicken stress SCS")
processor.push_to_hub(HF_ADAPTER_REPO, private=True, token=HF_WRITE_TOKEN,
                      commit_message="Processor (min/max_pixels utk resolusi training)")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/Arga23/chickenstress-vlm-lora/commit/51b653173b0ef8945508bd10f2ebdcaebfd3e441', commit_message='Processor (min/max_pixels utk resolusi training)', commit_description='', oid='51b653173b0ef8945508bd10f2ebdcaebfd3e441', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Arga23/chickenstress-vlm-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='Arga23/chickenstress-vlm-lora'), pr_revision=None, pr_num=None)

Data hasil skip/low-confidence/`undefined` bisa dikurasi lebih lanjut sebagai bahan fine-tuning
lanjutan, karena mencerminkan kondisi nyata lapangan yang mungkin belum terwakili di data pelatihan awal.